In [2]:
!pip install pandas
!pip install unicodedata

ERROR: Could not find a version that satisfies the requirement unicodedata (from versions: none)
ERROR: No matching distribution found for unicodedata


In [3]:
import pandas as pd
import unicodedata

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
df_ibge = pd.read_excel('/content/drive/MyDrive/Semantix/DadosTratamento/IBGE_Tratado.xlsx')
df_datasus = pd.read_excel('/content/drive/MyDrive/Semantix/DadosTratamento/DATASUS_Tratado.xlsx')

In [7]:
def normalizar_municipio(texto):
  if pd.isna(texto):
    return texto

  texto = str(texto).strip().upper()

  texto = unicodedata.normalize('NFKD', texto).encode('ASCII', 'ignore').decode('ASCII')
  return texto

In [8]:
df_ibge['Município'] = df_ibge['Município'].apply(normalizar_municipio)
df_datasus['Município'] = df_datasus['Município'].apply(normalizar_municipio)

In [9]:
df_consolidado = pd.merge(df_ibge, df_datasus, on="Município", how="left")

In [10]:
df_consolidado["Total"] = df_consolidado["Total"].fillna(0)

In [11]:
print("--- TESTE DE VALIDAÇÃO ---")
print("Total de municípios no arquivo final (deve ser 497):", len(df_consolidado))
print("Soma de casos no DATASUS original:", df_datasus['Total'].sum())
print("Soma de casos no arquivo final:   ", df_consolidado['Total'].sum())

--- TESTE DE VALIDAÇÃO ---
Total de municípios no arquivo final (deve ser 497): 497
Soma de casos no DATASUS original: 17697
Soma de casos no arquivo final:    17697.0


In [12]:
df_consolidado.to_excel('/content/drive/MyDrive/Semantix/dados_consolidados_RS.xlsx', index=False)

In [13]:
print("Suas colunas atuais são:", df_consolidado.columns.tolist())

Suas colunas atuais são: ['Município', 'Número Total de Mulheres (2022)', 'Total']


In [14]:
coluna_populacao = 'Número Total de Mulheres (2022)'

In [15]:
df_consolidado['Taxa_por_100k'] = (df_consolidado['Total'] / df_consolidado[coluna_populacao]) * 100000

In [16]:
df_consolidado['Taxa_por_100k'] = df_consolidado['Taxa_por_100k'].round(2)

In [17]:
print("\n--- OS 10 MUNICÍPIOS COM MAIOR NÚMERO ABSOLUTO DE CASOS ---")
print(df_consolidado.sort_values(by='Total', ascending=False)[['Município', 'Total', 'Taxa_por_100k']].head(10))


--- OS 10 MUNICÍPIOS COM MAIOR NÚMERO ABSOLUTO DE CASOS ---
           Município   Total  Taxa_por_100k
325     PORTO ALEGRE  1922.0         267.12
95     CAXIAS DO SUL  1189.0         497.07
201             IJUI  1011.0        2304.80
79            CANOAS   702.0         387.10
185         GRAVATAI   660.0         483.16
42   BENTO GONCALVES   586.0         923.47
364      SANTA MARIA   491.0         343.73
310          PELOTAS   469.0         270.22
398     SAO LEOPOLDO   433.0         383.05
231          LAJEADO   395.0         814.45


In [18]:
print("\n--- OS 10 MUNICÍPIOS COM MAIOR TAXA POR 100 MIL HABITANTES (CRÍTICO) ---")
print(df_consolidado.sort_values(by='Taxa_por_100k', ascending=False)[['Município', 'Total', 'Taxa_por_100k']].head(10))


--- OS 10 MUNICÍPIOS COM MAIOR TAXA POR 100 MIL HABITANTES (CRÍTICO) ---
          Município   Total  Taxa_por_100k
201            IJUI  1011.0        2304.80
62    CACIQUE DOBLE    42.0        1811.91
21    ARROIO DO SAL    94.0        1655.22
104         CHARRUA    22.0        1605.84
210          ITAARA    43.0        1547.88
116  CORONEL BARROS    21.0        1471.62
402    SAO MARTINHO    41.0        1467.43
54           BOZANO    15.0        1372.37
182         GRAMADO   265.0        1274.16
98     CERRO BRANCO    22.0        1169.59


In [20]:
df_consolidado.to_excel('/content/drive/MyDrive/Semantix/PROJETO_FINALCERTO_SEMANTIX_RS.xlsx', index=False)